## Step 5  
Inputs:  
- s3://thesis--ec331-s3/melted-price-bids/
- s3://thesis--ec331-s3/melted-volume-bids/  

Output: s3://thesis--ec331-s3/merged-price-volume-bids/  

Problems:
- I don't think that there is Settlement date in volume bids, I believe it is now trading bids

In [4]:
# Read in a test volume bid
import pandas as pd

# Define the file path
file_path = 's3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/enriched_volume_bids_20250320_085431_chunk1of5_part0001.parquet'

# Read the CSV file into a DataFrame, skipping the first row
volume_df = pd.read_parquet(file_path)

# Display general information about the DataFrame
print("DataFrame Info:")
volume_df.info()

# Show the first few rows of the DataFrame
volume_df.head()

# Show the last few rows of the DataFrame
print("\nDataFrame Tail:")
print(volume_df.tail())

# Display descriptive statistics for numerical columns
print("\nDataFrame Description:")
print(volume_df.describe(include='all'))

# List all column names
print("\nDataFrame Columns:")
print(volume_df.columns.tolist())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 39 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   I                             100000 non-null  string        
 1   BIDS                          100000 non-null  string        
 2   BIDOFFERPERIOD                100000 non-null  string        
 3   1                             100000 non-null  float64       
 4   DUID                          100000 non-null  object        
 5   BIDTYPE                       100000 non-null  object        
 6   TRADINGDATE                   100000 non-null  datetime64[ns]
 7   OFFERDATETIME                 100000 non-null  datetime64[ns]
 8   PERIODID                      100000 non-null  float64       
 9   MAXAVAIL                      100000 non-null  float64       
 10  FIXEDLOAD                     0 non-null       float64       
 11

In [2]:
# Assuming your DataFrame is named 'df'
# Convert TRADINGDATE column to datetime format for proper sorting
volume_df['TRADINGDATE'] = pd.to_datetime(volume_df['TRADINGDATE'])

# Get the minimum (earliest) and maximum (latest) dates
first_date = volume_df['TRADINGDATE'].min()
last_date = volume_df['TRADINGDATE'].max()

print(f"First date in dataset: {first_date}")
print(f"Last date in dataset: {last_date}")

First date in dataset: 2023-10-20 00:00:00
Last date in dataset: 2023-10-21 00:00:00


In [3]:
volume_df.head(5)

,I,BIDS,BIDOFFERPERIOD,1,DUID,BIDTYPE,TRADINGDATE,OFFERDATETIME,PERIODID,MAXAVAIL,...,Aggregation,Reg_Cap_generation__MW_,Max_Cap_generation__MW_,Max_ROC/Min_generation,Reg_Cap_consumption__MW_,Max_Cap_consumption__MW_,Max_ROC/Min_consumption,Comments,BIDBAND,BIDVOLUME
0,D,BIDS,BIDOFFERPERIOD,1.0,DRVIOT02,RAISE1SEC,2023-10-21,2023/10/21 04:42:52,242.0,1.0,...,None,None,None,None,NaN,NaN,NaN,NaN,1,1.0
1,D,BIDS,BIDOFFERPERIOD,1.0,DRVIOT02,RAISE1SEC,2023-10-21,2023/10/21 04:42:52,243.0,1.0,...,None,None,None,None,NaN,NaN,NaN,NaN,1,1.0
2,D,BIDS,BIDOFFERPERIOD,1.0,DRVIOT02,RAISE1SEC,2023-10-21,2023/10/21 04:42:52,244.0,1.0,...,None,None,None,None,NaN,NaN,NaN,NaN,1,1.0
3,D,BIDS,BIDOFFERPERIOD,1.0,DRVIOT02,RAISE1SEC,2023-10-21,2023/10/21 04:42:52,245.0,1.0,...,None,None,None,None,NaN,NaN,NaN,NaN,1,1.0
4,D,BIDS,BIDOFFERPERIOD,1.0,DRVIOT02,RAISE1SEC,2023-10-21,2023/10/21 04:42:52,246.0,1.0,...,None,None,None,None,NaN,NaN,NaN,NaN,1,1.0


In [5]:
# Read in a test price bid
import pandas as pd

# Define the file path
file_path = 's3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/deduped_part0001.parquet'

# Read the CSV file into a DataFrame, skipping the first row
df = pd.read_parquet(file_path)

# Display general information about the DataFrame
print("DataFrame Info:")
df.info()

# Show the first few rows of the DataFrame
df.head()

# Show the last few rows of the DataFrame
print("\nDataFrame Tail:")
print(df.tail())

# Display descriptive statistics for numerical columns
print("\nDataFrame Description:")
print(df.describe(include='all'))

# List all column names
print("\nDataFrame Columns:")
print(df.columns.tolist())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4350 entries, 0 to 4349
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   I                      4350 non-null   string        
 1   BIDS                   4350 non-null   string        
 2   BIDDAYOFFER            4350 non-null   string        
 3   1                      4350 non-null   float64       
 4   DUID                   4350 non-null   object        
 5   BIDTYPE                4350 non-null   object        
 6   SETTLEMENTDATE         4350 non-null   datetime64[ns]
 7   OFFERDATE              4350 non-null   datetime64[ns]
 8   VERSIONNO              4350 non-null   float64       
 9   PARTICIPANTID          4350 non-null   string        
 10  DAILYENERGYCONSTRAINT  0 non-null      float64       
 11  REBIDEXPLANATION       4340 non-null   string        
 12  MINIMUMLOAD            200 non-null    float64

In [19]:
import awswrangler as wr
import pandas as pd
import gc
import logging

logging.basicConfig(level=logging.INFO)

def main():
    # -------------------------------------------------------------------------
    # 1. Define S3 paths
    # -------------------------------------------------------------------------
    price_path  = "s3://thesis--ec331-s3/melted-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/deduped_part0001.parquet"
    volume_path = "s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/"
    
    # Extract the base folder name from volume_path for the output
    import os
    volume_folder_name = os.path.basename(volume_path.rstrip('/'))
    output_path = f"s3://thesis--ec331-s3/merged-price-volume-bids/{volume_folder_name}/"
    
    # -------------------------------------------------------------------------
    # 2. Load price data (once, since it's small)
    # -------------------------------------------------------------------------
    print("\n--- LOADING PRICE DATA ---")
    
    # Core columns we need from price data
    price_columns = ["DUID", "BIDTYPE", "SETTLEMENTDATE", "PARTICIPANTID", "BIDBAND", "BIDPRICE"]
    
    # Read price data
    price_df = wr.s3.read_parquet(
        path=price_path,
        columns=price_columns,
        use_threads=True
    )
    
    # Process price data
    price_df["SETTLEMENTDATE"] = pd.to_datetime(price_df["SETTLEMENTDATE"], errors="coerce")
    
    # Convert to optimal types for memory efficiency
    for col in ["DUID", "BIDTYPE", "PARTICIPANTID"]:
        price_df[col] = price_df[col].astype("category")
    price_df["BIDBAND"] = price_df["BIDBAND"].astype("category")
    price_df["BIDPRICE"] = price_df["BIDPRICE"].astype("float32")
    
    print(f"Price data loaded: {len(price_df):,} rows")
    
    # -------------------------------------------------------------------------
    # 3. Process volume files and merge with price data
    # -------------------------------------------------------------------------
    print("\n--- PROCESSING VOLUME FILES ---")
    
    # Core columns we need from volume data
    volume_columns = [
        "TRADINGDATE", "PERIODID", "MAXAVAIL",
        "DUID", "BIDBAND", "Participant", "Station_Name",
        "Region", "Dispatch_Type", "BIDVOLUME"
    ]
    
    # Get list of volume files
    volume_files = wr.s3.list_objects(path=volume_path, suffix=".parquet")
    print(f"Found {len(volume_files)} volume files to process")
    
    # Process each file
    total_merged_rows = 0
    
    for file_idx, file_path in enumerate(volume_files, 1):
        file_name = file_path.split('/')[-1]
        print(f"\nProcessing file {file_idx}/{len(volume_files)}: {file_name}")
        
        try:
            # Read volume data
            volume_df = wr.s3.read_parquet(
                path=file_path,
                columns=volume_columns,
                use_threads=True
            )
            
            volume_rows = len(volume_df)
            print(f"  -> Loaded {volume_rows:,} rows")
            
            # Process volume data
            volume_df["TRADINGDATE"] = pd.to_datetime(volume_df["TRADINGDATE"], errors="coerce")
            
            # Convert to optimal types
            for col in ["DUID", "Participant", "Station_Name", "Region", "Dispatch_Type", "PERIODID"]:
                volume_df[col] = volume_df[col].astype("category")
            volume_df["BIDBAND"] = volume_df["BIDBAND"].astype("category")
            volume_df["MAXAVAIL"] = volume_df["MAXAVAIL"].astype("float32")
            volume_df["BIDVOLUME"] = volume_df["BIDVOLUME"].astype("float32")
            
            # Merge with price data
            print("  -> Merging with price data...")
            merged_df = volume_df.merge(
                price_df,
                left_on=["TRADINGDATE", "DUID", "BIDBAND"],
                right_on=["SETTLEMENTDATE", "DUID", "BIDBAND"],
                how="left",
                suffixes=("_volume", "_price")
            )
            
            merged_rows = len(merged_df)
            total_merged_rows += merged_rows
            print(f"  -> Merged result: {merged_rows:,} rows")
            
            # Write to S3
            print(f"  -> Writing to S3...")
            wr.s3.to_parquet(
                df=merged_df,
                path=output_path,
                index=False,
                dataset=True,
                partition_cols=["TRADINGDATE"],
                mode="append"
            )
            
            # Clean up to save memory
            del volume_df, merged_df
            gc.collect()
            
        except Exception as e:
            print(f"Error processing file {file_name}: {str(e)}")
            continue
    
    print(f"\n--- COMPLETED: Total merged rows: {total_merged_rows:,} ---")
    print(f"Data written to: {output_path}")

if __name__ == "__main__":
    main()


--- LOADING PRICE DATA ---
Price data loaded: 4,350 rows

--- PROCESSING VOLUME FILES ---
Found 5 volume files to process

Processing file 1/5: enriched_volume_bids_20250320_085431_chunk1of5_part0001.parquet
  -> Loaded 100,000 rows
  -> Merging with price data...
  -> Merged result: 100,000 rows
  -> Writing to S3...

Processing file 2/5: enriched_volume_bids_20250320_085431_chunk2of5_part0001.parquet
  -> Loaded 100,000 rows
  -> Merging with price data...
  -> Merged result: 100,000 rows
  -> Writing to S3...

Processing file 3/5: enriched_volume_bids_20250320_085431_chunk3of5_part0001.parquet
  -> Loaded 100,000 rows
  -> Merging with price data...
  -> Merged result: 100,000 rows
  -> Writing to S3...

Processing file 4/5: enriched_volume_bids_20250320_085431_chunk4of5_part0001.parquet
  -> Loaded 100,000 rows
  -> Merging with price data...
  -> Merged result: 100,000 rows
  -> Writing to S3...

Processing file 5/5: enriched_volume_bids_20250320_085431_chunk5of5_part0001.parquet

In [11]:

# Add this before your existing code to check the actual schema
price_path  = "s3://thesis--ec331-s3/de-duped-price-bids/RAISE1SEC_PUBLIC_DVD_BIDDAYOFFER_202310010000.parquet/"

schema_test = wr.s3.read_parquet_metadata(path=price_path)
schema_test.columns

AttributeError: '_ReadTableMetadataReturnValue' object has no attribute 'columns'